In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from odtlearn.flow_oct import FlowOCT, BendersOCT
from odtlearn.utils.binarize import Binarizer

# `FlowOCT` Examples

## Example 0: Binarization

The following example shows how to binarize a dataset with categorical, integer, and continuous features using the built-in `Binarizer` class. This class follows the scikit-learn fit-transform paradigm, making it easy to integrate into your preprocessing pipeline.


In [ ]:
number_of_child_list = [1, 2, 4, 3, 1, 2, 4, 3, 2, 1]
age_list = [10, 20, 40, 30, 10, 20, 40, 30, 20, 10]
race_list = [
    "Black",
    "White",
    "Hispanic",
    "Black",
    "White",
    "Black",
    "White",
    "Hispanic",
    "Black",
    "White",
]
sex_list = ["M", "F", "M", "M", "F", "M", "F", "M", "M", "F"]
income_list = [50000, 75000, 100000, 60000, 80000, 55000, 90000, 70000, 85000, 65000]

df = pd.DataFrame(
    list(zip(sex_list, race_list, number_of_child_list, age_list, income_list)),
    columns=["sex", "race", "num_child", "age", "income"],
)

print(df)

In [ ]:
binarizer = Binarizer(
    categorical_cols=["sex", "race"],
    integer_cols=["num_child", "age"],
    real_cols=["income"],
    n_bins=5  # Number of bins for continuous features
)

# Fit and transform the data
df_enc = binarizer.fit_transform(df)

print(df_enc)

The `Binarizer` class follows the scikit-learn fit-transform paradigm:

We initialize the `Binarizer` with our desired parameters, specifying which columns are categorical, integer, and real-valued.
We call the fit_transform method, which first fits the binarizer to our data (learning the necessary encoding schemes) and then transforms the data using those learned encodings.

The resulting `df_enc` DataFrame contains the binarized version of our original data:

Categorical columns (sex, race) are one-hot encoded.
Integer columns (num_child, age) are binary encoded, where each column represents "greater than or equal to" a certain value.
The continuous column (income) is first discretized into 5 bins, then binary encoded similar to the integer columns.

In [ ]:
print(df_enc.columns)

This binarized data is now ready to be used with any of the models in ODTlearn, which require binary input features.
If you need to transform new data using the same encoding scheme, you can use the transform method of the fitted binarizer:

In [ ]:
new_data = pd.DataFrame({
    "sex": ["F", "M"],
    "race": ["Hispanic", "White"],
    "num_child": [2, 3],
    "age": [20, 30],
    "income": [70000, 80000]
})

new_data_enc = binarizer.transform(new_data)
print(new_data_enc)

Note that for categorical features, if the new data contains categories not seen during fitting, the transform method will raise an error. In such cases, you might need to refit the binarizer on a dataset that includes all possible categories.

## Example 1: Varying `depth` and `_lambda`
In this part, we study a simple example and investigate different parameter combinations to provide intuition on how they affect the structure of the tree.

First we generate the data for our example. The diagram within the code block shows the training dataset. Our dataset has two binary features (X1 and X2) and two class labels (+1 and -1).

In [ ]:
from odtlearn.datasets import flow_oct_example

"""
    X2
    |               |
    |               |
    1    + +        |    -
    |               |   
    |---------------|-------------
    |               |
    0    - - - -    |    + + +
    |    - - -      |
    |______0________|_______1_______X1
"""


X, y = flow_oct_example()

### Tree with `depth = 1`

In the following, we fit a classification tree of depth 1, i.e., a tree with a single branching node and two leaf nodes.

In [ ]:
stcl = FlowOCT(depth=1, solver="cbc", time_limit=100)
stcl.fit(X, y)

In [ ]:
predictions = stcl.predict(X)
print(f'Optimality gap is {stcl.optim_gap}')
print(f"In-sample accuracy is {np.sum(predictions==y)/y.shape[0]}")

Users can access statistics from the optimization run such as optimality gap, number of nodes, number of constraints, etc. Directly as properties of the initialized class or through the `_solver` object. For example, one can access the optimality gap and the number of solutions after fitting the optimal decision tree through the `optim_gap` and `num_solutions` properties.

In [ ]:
print(f'Optimality gap is {stcl.optim_gap}')
print(f'Number of solutions {stcl.num_solutions}')

As we can see above, we find the optimal tree and the in-sample accuracy is 76%.

ODTlearn provides two different ways of visualizing the structure of the tree. The first method prints the structure of the tree in the console:

In [ ]:
stcl.print_tree()

The second method plots the structure of the tree using `matplotlib`:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
stcl.plot_tree(ax=ax)
plt.show()

In [ ]:
stcl_progress_log = FlowOCT(depth=1, solver="cbc", time_limit=100)
stcl_progress_log.fit(X, y)

### Tree with `depth = 2`

Now we increase the depth of the tree to achieve higher accuracy.

In [ ]:
stcl = FlowOCT(depth=2, solver="cbc")
stcl.fit(X, y)

In [ ]:
predictions = stcl.predict(X)
print(f"In-sample accuracy is {np.sum(predictions==y)/y.shape[0]}")

As we can see, with depth 2, we can achieve 100% in-sample accuracy.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
stcl.plot_tree(ax=ax, fontsize=20)
plt.show()

### Tree with `depth=2` and Positive `_lambda`

As we saw in the above example, with depth 2, we can fully classify the training data. However if we add a regularization term with a high enough value of `_lambda`, we can justify pruning one of the branching nodes to get a sparser tree. In the following, we observe that as we increase `_lambda` from 0 to 0.51, one of the branching nodes gets pruned and as a result, the in-sample accuracy drops to 92%.

In [ ]:
stcl = FlowOCT(solver="cbc", depth=2, _lambda=0.51)
stcl.fit(X, y)

In [ ]:
predictions = stcl.predict(X)
print(f"In-sample accuracy is {np.sum(predictions==y)/y.shape[0]}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
stcl.plot_tree(ax=ax, fontsize=20)
plt.show()

## Example 2: Different Objective Functions

In the following, we have a toy example with an imbalanced data, with the positive class being the minority class.

In [ ]:
'''
    X2
    |               | 
    |               |
    1    + - -      |    -
    |               |   
    |---------------|--------------
    |               |
    0    - - - +    |    - - -
    |    - - - -    |
    |______0________|_______1_______X1
'''
X = np.array([[0,0],[0,0],[0,0],[0,0],[0,0],[0,0],[0,0],[0,0],
              [1,0],[1,0],[1,0],
              [1,1],
              [0,1],[0,1],[0,1]])
y = np.array([0,0,0,0,0,0,0,1,
              0,0,0,
              0,
              1,0,0])

### Tree with classification accuracy objective

In [ ]:
stcl_acc = FlowOCT(solver="cbc", depth=2, obj_mode="acc")
stcl_acc.fit(X, y)


In [ ]:
predictions = stcl_acc.predict(X)
print(f"In-sample accuracy is {np.sum(predictions==y)/y.shape[0]}")

In [ ]:
stcl_acc.print_tree()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5)) 
stcl_acc.plot_tree(ax=ax, fontsize=20)
plt.show()

### Tree with Balanced Classification Accuracy Objective

In [ ]:
stcl_balance = FlowOCT(
    solver="cbc",
    depth=2,
    obj_mode="balance",
    _lambda=0,
    verbose=False,
)
stcl_balance.fit(X, y)
predictions = stcl_balance.predict(X)
print(f"In-sample accuracy is {np.sum(predictions==y)/y.shape[0]}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5)) 
stcl_balance.plot_tree(ax=ax, fontsize=20)
plt.show()

As we can see, when we maximize accuracy, i.e., when `obj_mode = 'acc'`, the optimal tree is just a single node without branching, predicting the majority class for the whole dataset. But when we change the objective mode to balanced accuracy, we account for the minority class by sacrificing the overal accuracy.

## Example 3: UCI Data Example

In this section, we fit a tree of depth 3 on a real world dataset called the [`balance` dataset](https://archive.ics.uci.edu/ml/datasets/Balance+Scale) from the UCI Machine Learning repository. 

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from odtlearn.datasets import balance_scale_data

In [ ]:
# read data
data = balance_scale_data()
print(f"shape{data.shape}")
data.columns

In [ ]:
y = data.pop("target")

X_train, X_test, y_train, y_test = train_test_split(
    data, y, test_size=0.33, random_state=42
)

In [ ]:
stcl = BendersOCT(solver="cbc", depth=3, time_limit=200, obj_mode="acc", verbose=True)
stcl.store_search_progress_log = True
stcl.fit(X_train, y_train)

In [ ]:
stcl.print_tree()

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
stcl.plot_tree(ax=ax, fontsize=20, color_dict={"node": None, "leaves": []})
plt.show()

In [ ]:
test_pred = stcl.predict(X_test)
print('The out-of-sample accuracy is {}'.format(np.sum(test_pred==y_test)/y_test.shape[0]))

We also provide a simple function allowing users to plot the search progress log over time. Note that you must set the attribute `store_search_progress_log` to `True` before calling the `fit` method to ensure that the bound information is stored. 

In [ ]:
stcl.plot_search_progress()

## Example 4: User-defined Weights

In this example, we'll demonstrate how to use user-defined weights with FlowOCT and BendersOCT. We'll use a small binary classification dataset and show how user-defined weights can affect the learned tree.

First, let's create our small binary classification dataset:

In [ ]:
np.random.seed(42)
X = np.random.randint(0, 2, size=(20, 5))
y = np.random.randint(0, 2, size=20)

print("Dataset shape:", X.shape)
print("Class distribution:", np.bincount(y))

Now, let's create weights that heavily favor class 1:

In [ ]:
weights = np.ones_like(y)
weights[y == 1] = 10

print("Weight distribution:")
print("Class 0:", weights[y == 0].mean())
print("Class 1:", weights[y == 1].mean())

### FlowOCT with User-defined Weights
Let's fit a FlowOCT model with user-defined weights and compare it to a model without the accuracy objective:

In [ ]:
# FlowOCT without custom weights
flow_oct_default = FlowOCT(solver="cbc", obj_mode="acc", depth=2, time_limit=10)
flow_oct_default.fit(X, y)

# FlowOCT with custom weights
flow_oct_custom = FlowOCT(solver="cbc", obj_mode="weighted", depth=2, time_limit=10)
flow_oct_custom.fit(X, y, weights=weights)

print("Default FlowOCT predictions:", flow_oct_default.predict(X))
print("User-defined weights FlowOCT predictions:", flow_oct_custom.predict(X))

print("Default FlowOCT accuracy:", (flow_oct_default.predict(X) == y).mean())
print("User-defined weights FlowOCT accuracy:", (flow_oct_custom.predict(X) == y).mean())
print("User-defined weights FlowOCT weighted accuracy:", np.average(flow_oct_custom.predict(X) == y, weights=weights))

Let's visualize both trees:

In [ ]:

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

flow_oct_default.plot_tree(ax=ax1, fontsize=10)
ax1.set_title("Default FlowOCT")

flow_oct_custom.plot_tree(ax=ax2, fontsize=10)
ax2.set_title("User-defined Weights FlowOCT")

plt.tight_layout()
plt.show()

### BendersOCT with User-defined Weights
Now let's do the same with BendersOCT:

In [ ]:
# BendersOCT without custom weights
benders_oct_default = BendersOCT(solver="cbc", obj_mode="acc", depth=2, time_limit=10)
benders_oct_default.fit(X, y)

# BendersOCT with custom weights
benders_oct_custom = BendersOCT(solver="cbc", obj_mode="weighted", depth=2, time_limit=10, verbose=False)
benders_oct_custom.fit(X, y, weights=weights)

print("Default BendersOCT predictions:", benders_oct_default.predict(X))
print("User-defined weights BendersOCT predictions:", benders_oct_custom.predict(X))

print("Default BendersOCT accuracy:", (benders_oct_default.predict(X) == y).mean())
print("User-defined weights BendersOCT accuracy:", (benders_oct_custom.predict(X) == y).mean())
print("User-defined weights BendersOCT weighted accuracy:", np.average(benders_oct_custom.predict(X) == y, weights=weights))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

benders_oct_default.plot_tree(ax=ax1, fontsize=10)
ax1.set_title("Default BendersOCT")

benders_oct_custom.plot_tree(ax=ax2, fontsize=10)
ax2.set_title("User-defined Weights BendersOCT")

plt.tight_layout()
plt.show()

In this example, we've demonstrated how to use user-defined weights with both FlowOCT and BendersOCT. By setting `obj_mode="weighted"` and providing weights during the `fit` method call, we can influence the importance of different samples in the training process.
The user-defined weights in this example heavily favor class 1, which may result in trees that are more likely to predict class 1, potentially at the cost of overall accuracy. However, this can be useful in scenarios where misclassifying one class is more costly than misclassifying the other, or when dealing with imbalanced datasets.
Note that the actual results may vary due to the random nature of the dataset and the optimization process. You may want to run the code multiple times or with different random seeds to get a better understanding of the effects of user-defined weights.

## References
* Dua, D. and Graff, C. (2019). [UCI Machine Learning Repository](http://archive.ics.uci.edu/ml). Irvine, CA: University of California, School of Information and Computer Science.
* Aghaei, S., Gómez, A., & Vayanos, P. (2025). Strong optimal classification trees. *Operations Research*, 73(4), 2223-2241.